In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/clean/clean_data.csv")

df.columns

Index(['marital_status', 'application_mode', 'application_order', 'course',
       'daytime_evening_attendance', 'previous_qualification',
       'previous_qualification_grade', 'nationality', 'mothers_qualification',
       'fathers_qualification', 'mothers_occupation', 'fathers_occupation',
       'admission_grade', 'displaced', 'educational_special_needs', 'debtor',
       'tuition_fees_up_to_date', 'gender', 'scholarship_holder',
       'age_at_enrollment', 'international',
       'curricular_units_1st_sem_credited',
       'curricular_units_1st_sem_enrolled',
       'curricular_units_1st_sem_evaluations',
       'curricular_units_1st_sem_approved', 'curricular_units_1st_sem_grade',
       'curricular_units_1st_sem_without_evaluations',
       'curricular_units_2nd_sem_credited',
       'curricular_units_2nd_sem_enrolled',
       'curricular_units_2nd_sem_evaluations',
       'curricular_units_2nd_sem_approved', 'curricular_units_2nd_sem_grade',
       'curricular_units_2nd_sem_wit

In [2]:
binary_features = [
    "scholarship_holder",
    "international"
]

numerical_features = [
    "age_at_enrollment",
    "curricular_units_1st_sem_credited",
    "curricular_units_1st_sem_enrolled",
    "curricular_units_1st_sem_evaluations",
    "curricular_units_1st_sem_approved",
    "curricular_units_1st_sem_grade",
    "curricular_units_1st_sem_without_evaluations"
]

my_columns = [
    "scholarship_holder",
    "age_at_enrollment",
    "international",
    "curricular_units_1st_sem_credited",
    "curricular_units_1st_sem_enrolled",
    "curricular_units_1st_sem_evaluations",
    "curricular_units_1st_sem_approved",
    "curricular_units_1st_sem_grade",
    "curricular_units_1st_sem_without_evaluations",
    "target"
]

df_features = df[my_columns].copy()

df_features.head()

,scholarship_holder,age_at_enrollment,international,curricular_units_1st_sem_credited,curricular_units_1st_sem_enrolled,curricular_units_1st_sem_evaluations,curricular_units_1st_sem_approved,curricular_units_1st_sem_grade,curricular_units_1st_sem_without_evaluations,target
0,0,20,0,0,0,0,0,0.000000,0,Dropout
1,0,19,0,0,6,6,6,14.000000,0,Graduate
2,0,19,0,0,6,0,0,0.000000,0,Dropout
3,0,20,0,0,6,8,6,13.428571,0,Graduate
4,0,45,0,0,6,9,5,12.333333,0,Graduate


## Feature Engineering

### Feature Creation - Approval Rate

The approval rate measures the proportion of enrolled
curricular units successfully approved by a student.

This feature accounts for differences in academic
workload between students.

In [3]:
df_features[
    df_features["curricular_units_1st_sem_approved"] >
    df_features["curricular_units_1st_sem_enrolled"]
].shape[0]

0

In [4]:
#zero enrolled units
zero_enrolled = df_features[
    df_features["curricular_units_1st_sem_enrolled"] == 0
]

zero_enrolled.describe().T

,count,mean,std,min,25%,50%,75%,max
scholarship_holder,180.0,0.244444,0.430956,0.0,0.0,0.0,0.0,1.0
age_at_enrollment,180.0,20.138889,3.579526,17.0,18.0,19.0,20.0,38.0
international,180.0,0.033333,0.180006,0.0,0.0,0.0,0.0,1.0
curricular_units_1st_sem_credited,180.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
curricular_units_1st_sem_enrolled,180.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
curricular_units_1st_sem_evaluations,180.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
curricular_units_1st_sem_approved,180.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
curricular_units_1st_sem_grade,180.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0
curricular_units_1st_sem_without_evaluations,180.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0


In [5]:
df_features["approval_rate_1st_sem"] = (
    df_features["curricular_units_1st_sem_approved"] /
    df_features["curricular_units_1st_sem_enrolled"].replace(0, np.nan)
)

df_features["approval_rate_1st_sem"].describe()

count    4244.000000
mean        0.727484
std         0.342821
min         0.000000
25%         0.600000
50%         0.857143
75%         1.000000
max         1.000000
Name: approval_rate_1st_sem, dtype: float64

In [6]:
df_features["approval_rate_1st_sem"].isna().sum()

np.int64(180)

### Approval Rate - Observations

- Created a new feature to measure academic performance
  relative to the number of enrolled curricular units
- The average approval rate is 72.75%
- The median approval rate is 85.71%
- No students have more approved units than enrolled units
- 180 students have zero enrolled units, resulting in
  undefined approval rates (NaN)
- These students will be retained, and the handling of
  undefined approval rates will be discussed with the team

## Feature Transformation

Numerical features will be standardised for models
that are sensitive to differences in feature scales.

Binary features will retain their original 0 and 1
encoding.

Scaling will be fitted on training data only as part
of the shared Machine Learning pipeline.

In [7]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

In [8]:
numerical_pipeline = Pipeline([
    ("scaling", StandardScaler())
])

In [9]:
preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("binary", "passthrough", binary_features)
])

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical', ...), ('binary', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

### Preprocessing Decisions

- StandardScaler will be applied to the seven
  original numerical features for scale-sensitive
  Machine Learning models.

- Scholarship holder and international status
  will retain their binary encoding.

- No categorical grouping or OneHotEncoding
  is required for the assigned features.

- The newly created approval rate is not yet
  included in the preprocessing pipeline due
  to its undefined values.

- The scaler will be fitted during model training
  using training data only.

- StandardScaler was initially proposed for the assigned
  numerical features. The consolidated preprocessing
  pipeline uses MinMaxScaler instead.

- All nine original assigned features are included in
  the consolidated pipeline.

- The approval rate remains a proposed additional
  feature to be evaluated during model development.

## Final Feature Engineering Recommendations

| Feature | Recommendation |
|---|---|
| Scholarship holder | Keep binary encoding |
| Age at enrollment | Keep and scale |
| International | Keep binary encoding |
| Credited units | Keep and scale |
| Enrolled units | Keep and scale |
| Evaluations | Keep and scale |
| Approved units | Keep and scale |
| Grade | Keep and scale |
| Without evaluations | Keep and scale |
| Approval rate | New feature proposed for modelling |

Scaling is recommended for scale-sensitive models.

No original features or observations have been removed.

Valid zero values have been retained.

The approval rate is recommended as an additional
feature, subject to agreement on handling undefined
values and evaluation during model development.

The team has agreed on an 80/20 stratified train/test
split using RANDOM_STATE = 42.

Enrolled students are excluded from model training
and testing and retained separately for future
dropout risk predictions.

The timing of first-semester information must also
be considered when defining the prediction point.